# MOZYME en GPU: termoquímica (FORCE / FORCETS + THERMO), CPU frente a GPU

Validación antes de documentarla en el README. MOPAC calcula la termoquímica a partir de las frecuencias armónicas;
`FORCE` construye la hessiana por diferencias finitas del gradiente (2 gradientes por coordenada, 6N en total) y
`FORCETS` solo la de los átomos marcados para optimizar (hessiana parcial, con `OPT("cadena+residuo"=radio)`).
Cada gradiente es un SCF + gradiente de MOZYME, justo lo que la GPU acelera.

Qué hace (`tests/check_mozyme_gpu_thermo.py`):

1. **Chignolina (1UAO, 140 átomos con H)**: prepara el PDB (primer modelo de RMN, `ADD-H`), la optimiza en GPU hasta
   `GNORM=1` y en esa geometría corre `FORCE THERMO(298)` en CPU y en GPU: compara las 414 frecuencias, la energía
   de punto cero, H, S y G = H − TS. Opcional: la GPU con `PRECISE`.
2. **Hessiana parcial**: `FORCETS THERMO(298) OPT("A9"=4)` (átomos a menos de 4 Å del residuo 9 de la cadena A, el
   Trp), CPU y GPU.
3. **Crambina (1CRN, 642 átomos), solo GPU**: optimización, `FORCE THERMO` completa (3 852 gradientes) y
   `FORCETS` parcial, para medir el tiempo real.

Tolerancias provisionales: ZPE ±0.1 kcal/mol, S ±1 cal/(mol·K), G ±0.3 kcal/mol, RMS de las frecuencias
≥ 100 cm⁻¹ ≤ 2 cm⁻¹. Corrección incluida: en `FORCETS` MOPAC ponderaba cada coordenada con la masa del átomo
equivocado cuando la selección no eran los primeros átomos (frecuencias parciales de hasta 6600 cm⁻¹); corregido en
`src/forces/freqcy.F90`.

Punto estacionario: con `GNORM=1` la chignolina conserva una frecuencia imaginaria (−166 cm⁻¹ en CPU), es decir, no es
un mínimo verdadero. La comparación CPU/GPU es válida igual (misma geometría), pero para una termoquímica con sentido
físico baja `GNORM` (0.1–0.3) hasta que no queden frecuencias imaginarias.

Primera corrida (A100): FORCE 26× más rápida que la CPU y CPU/GPU casi iguales con el criterio por defecto. Pero al
compararlo con una referencia independiente apareció un error grande del MOPAC original, no de la GPU: con el
criterio SCF por defecto de MOZYME (SCFCRT = 0.01 kcal/mol) cada gradiente desplazado queda sin converger y la
hessiana sale rígida. Frente a la FORCE del SCF convencional (sin MOZYME, convergido de verdad), en la chignolina:

| MOZYME en CPU | ZPE | S | G = ZPE + H − TS | modos 0–500 cm⁻¹ |
|---|---:|---:|---:|---:|
| SCFCRT = 0.01 (por defecto) | +53.0 kcal/mol | −130 cal/(mol·K) | +78.9 kcal/mol | +124 cm⁻¹ |
| SCFCRT = 0.001 | +25.3 | −82 | +42.6 | +67 cm⁻¹ |
| SCFCRT = 0.0001 | +0.02 | −0.08 | +0.03 | +0.1 cm⁻¹ |

Con 0.0001 MOZYME coincide con la referencia, pero en CPU cuesta 37× más. Esta versión: (1) la referencia
convencional en CPU, (2) el barrido MOZYME en GPU con SCFCRT = 0.01, 0.001 y 0.0001 comparado con ella (coste y
precisión de cada criterio en la GPU), (3) CPU frente a GPU con el criterio por defecto y la GPU dos veces.

Aceleración de la hessiana precisa (esta versión): (1) `ITRY_SCAN` mide cuántas iteraciones de SCF hacen falta
por geometría desplazada para igualar la referencia (con un criterio inalcanzable cada SCF hace exactamente ITRY
iteraciones); (2) `PARALLEL_WORKERS` calcula la misma hessiana con varios procesos MOPAC a la vez en la GPU
(`scripts/mozyme_parallel_force.py`: cada proceso calcula un rango de filas con `FORCE RESTART CYCLES=n` y las
matrices parciales se suman), y la compara con la secuencial en tiempo y resultado.

Tiempo: compilación ~10 min; parte pequeña ~10 min (la CPU domina); crambina ~20–30 min en una A100.

## 1. GPU

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

## 2. Código fuente

In [ ]:
from pathlib import Path
import shutil, subprocess

REPO_URL = 'https://github.com/juvenalyosa/mopac_gpu.git'
BRANCH = 'main'
CONTENT = Path('/content')
SRC = CONTENT / 'mopac_src'
BUILD = CONTENT / 'mopac_build'
if SRC.exists():
    shutil.rmtree(SRC)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(SRC)], check=True)
print(subprocess.run(['git', '-C', str(SRC), 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

## 3. Compilar MOPAC con GPU

In [ ]:
import subprocess, shutil

def run(cmd, **kw):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    return subprocess.run([str(c) for c in cmd], check=True, **kw)

pkgs = ['cmake', 'gfortran', 'ninja-build', 'libblas-dev', 'liblapack-dev']
for attempt in range(2):
    subprocess.run(['apt-get', 'update', '-qq'], check=False)
    r = subprocess.run(['apt-get', 'install', '-y', '-qq', '--no-install-recommends', *pkgs],
                       check=False, capture_output=True, text=True)
    if r.returncode == 0:
        break
    if attempt == 1:
        raise SystemExit('apt-get install failed twice')
if BUILD.exists():
    shutil.rmtree(BUILD)
cmake_cmd = ['cmake', '-S', SRC, '-B', BUILD, '-GNinja', '-DGPU=ON', '-DTESTS=OFF', '-DCMAKE_BUILD_TYPE=RelWithDebInfo']
try:
    run(cmake_cmd + ['-DCUDA_ARCHS=native'])
except subprocess.CalledProcessError:
    shutil.rmtree(BUILD, ignore_errors=True)
    run(cmake_cmd + ['-DCUDA_ARCHS=all'])
print(subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout.strip().splitlines()[-1:])
print('$ cmake --build', BUILD, '--target mopac --parallel 2', flush=True)
r = subprocess.run(['cmake', '--build', str(BUILD), '--target', 'mopac', '--parallel', '2'],
                   capture_output=True, text=True)
if r.returncode != 0:
    log = (r.stdout + r.stderr).splitlines()
    errors = [l for l in log if 'error' in l.lower() or 'Error' in l]
    print('\n'.join(errors[:60]) if errors else '\n'.join(log[-80:]))
    raise SystemExit('build failed: compiler messages above')
MOPAC = BUILD / 'mopac'
assert MOPAC.exists(), 'mopac executable not built'
print('OK:', MOPAC)

## 3b. MPS (opcional): varios procesos compartiendo la GPU

La hessiana paralela lanza varios procesos MOPAC sobre la misma GPU. Sin MPS la GPU los atiende por turnos; con el
daemon MPS de NVIDIA sus kernels se ejecutan a la vez. Si falla (algunos entornos no lo permiten), la hessiana
paralela funciona igual, solo con menos ganancia.

In [ ]:
import subprocess
r = subprocess.run(['nvidia-cuda-mps-control', '-d'], capture_output=True, text=True)
print('MPS:', 'activo' if r.returncode == 0 else f'no disponible ({r.stderr.strip() or r.returncode})')

## 4. Parámetros y ejecución

La salida se imprime a medida que avanza. `REUSE=True` reanaliza las salidas ya calculadas sin volver a correr MOPAC.

In [ ]:
PDB_ID = '1UAO'           # sistema pequeño, CPU y GPU (la GPU se activa con más de 100 átomos)
PARTIAL = 'A9=4'          # FORCETS: cadena+residuo = radio en Å
LARGE_PDB_ID = '1CRN'     # sistema grande, solo GPU
LARGE_PARTIAL = 'A25=5'
TEMPERATURE = 298
GNORM = 1.0               # la termoquímica necesita un punto estacionario
PRECISE = False           # PRECISE no cambia FORCE (comprobado: mismo resultado bit a bit en CPU)
SCFCRT = None             # None = criterio por defecto de MOZYME (0.01) en la comparación CPU/GPU (barata)
SCFCRT_SCAN = '0.01,0.001'         # criterio por defecto y 10x más estricto, frente a la referencia
ITRY_SCAN = '30,60,100,200'        # SCFCRT inalcanzable: cada SCF hace exactamente ITRY iteraciones
PARALLEL_WORKERS = 4               # la misma hessiana con 4 procesos MOPAC a la vez en la GPU
PARALLEL_ITRY = 60                 # ITRY de la hessiana paralela (compárala con ITRY_SCAN)
LARGE_WORKERS = 0                  # >0: crambina precisa en paralelo (larga; prueba con 6 en una A100)
GPU_REPEAT = 2            # corridas GPU de FORCE, para medir la variación entre corridas
SKIP_LARGE = False
REUSE = False

import subprocess, sys
WORK = CONTENT / 'mozyme_gpu_thermo'
cmd = [sys.executable, str(SRC / 'tests/check_mozyme_gpu_thermo.py'), str(MOPAC), '--pdb-id', PDB_ID,
       '--partial', PARTIAL, '--large-pdb-id', LARGE_PDB_ID, '--large-partial', LARGE_PARTIAL,
       '--temperature', str(TEMPERATURE), '--gnorm', str(GNORM), '--work-dir', str(WORK)]
if PRECISE: cmd.append('--precise')
if SCFCRT: cmd += ['--scfcrt', str(SCFCRT)]
cmd += ['--gpu-repeat', str(GPU_REPEAT)]
if SCFCRT_SCAN: cmd += ['--scfcrt-scan', SCFCRT_SCAN]
if ITRY_SCAN: cmd += ['--itry-scan', ITRY_SCAN]
if PARALLEL_WORKERS: cmd += ['--parallel-workers', str(PARALLEL_WORKERS), '--parallel-itry', str(PARALLEL_ITRY)]
if LARGE_WORKERS: cmd += ['--large-workers', str(LARGE_WORKERS)]
if SKIP_LARGE: cmd.append('--skip-large')
if REUSE: cmd.append('--reuse')
print('$', ' '.join(cmd), flush=True)
proc = subprocess.Popen(cmd, cwd=SRC, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
print('exit status', proc.wait())

## 5. Frecuencias CPU frente a GPU

Cada punto es un modo normal (ordenados por frecuencia). Los modos por debajo de ~100 cm⁻¹ son los más sensibles al
ruido del SCF (la hessiana divide diferencias de gradiente por un desplazamiento de ~0.004 Å).

In [ ]:
sys.path.insert(0, str(SRC / 'tests'))
import importlib, check_mozyme_gpu_thermo as th
importlib.reload(th)
import matplotlib.pyplot as plt

def freqs(sub):
    tag = f'_scfcrt{SCFCRT:g}' if SCFCRT else ''
    out = WORK / PDB_ID / (sub + tag) / (('forcets' if 'forcets' in sub else 'force') + '.out')
    return sorted(th.frequencies(out.read_text(errors='ignore'))) if out.exists() else []

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for sub, ref_sub, label in (('force_gpu', 'force_cpu', 'FORCE GPU'),
                            ('force_gpu_precise', 'force_cpu', 'FORCE GPU PRECISE'),
                            ('forcets_gpu', 'forcets_cpu', 'FORCETS GPU')):
    ref, cur = freqs(ref_sub), freqs(sub)
    if ref and len(ref) == len(cur):
        diff = [g - c for c, g in zip(ref, cur)]
        ax[0].plot(ref, diff, '.', ms=3, label=label)
        ax[1].hist(diff, bins=60, alpha=0.5, label=label)
ax[0].axhline(0, c='gray', lw=0.5); ax[0].set_xlabel('frecuencia CPU (cm⁻¹)'); ax[0].set_ylabel('GPU − CPU (cm⁻¹)')
ax[0].legend(); ax[1].set_xlabel('GPU − CPU (cm⁻¹)'); ax[1].legend(); plt.tight_layout(); plt.show()

## 6. Descargar las salidas

In [ ]:
import shutil
archive = shutil.make_archive(str(CONTENT / 'mozyme_gpu_thermo'), 'zip', WORK)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print('descarga manual:', exc)